# NEDI x2 Colab Pilot

This notebook runs one Set5 image through native NEDI x2. It is a correctness and runtime pilot, not the final NEDI evaluation.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/divinesta/SuperResolution-Comparative-Analysis.git'
REPO_ROOT = Path('/content/SuperResolution-Comparative-Analysis')

if REPO_ROOT.exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)

os.chdir(REPO_ROOT)
print(f'Repository ready: {REPO_ROOT}')

In [ ]:
import sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements.txt')],
    check=True,
)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.evaluation.nedi import NEDIEvaluationConfig, evaluate_nedi_dataset
from app.evaluation.experiment import write_results_csv
print('NEDI evaluator imported successfully.')

In [ ]:
from datetime import UTC, datetime

from app.evaluation.data_validation import validate_prepared_dataset

DATA_ROOT = Path('/content/drive/MyDrive/FYP_SR_Data')
RUN_ID = datetime.now(UTC).strftime('%Y%m%d_%H%M%S_utc')
RUN_ROOT = DATA_ROOT / 'results' / 'final_nedi' / 'pilot' / RUN_ID
METRICS_ROOT = RUN_ROOT / 'metrics'
IMAGES_ROOT = RUN_ROOT / 'images'

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'Dataset root not found: {DATA_ROOT}')

validation = validate_prepared_dataset('Set5', 2, DATA_ROOT)
print(f'VALID: {validation.dataset} x{validation.scale} has {validation.image_count} pairs.')
print(f'Pilot output: {RUN_ROOT}')

In [ ]:
# A pilot uses fewer timings for a quick runtime estimate.
# The final NEDI run will use 3 warm-ups and 10 timed runs.
config = NEDIEvaluationConfig(
    dataset='Set5',
    scale=2,
    window_size=8,
    edge_threshold=8.0,
    warmup_runs=1,
    timed_runs=3,
)
records = evaluate_nedi_dataset(
    validation.hr_directory,
    validation.lr_directory,
    config,
    sr_output_dir=IMAGES_ROOT,
    max_images=1,
)
output_csv = write_results_csv(records, METRICS_ROOT / 'Set5_x2_nedi_pilot.csv')
print(f'Pilot result saved to: {output_csv}')

In [ ]:
record = records[0]
for field in (
    'image', 'psnr_y', 'ssim_y', 'psnr_rgb', 'ssim_rgb',
    'latency_mean_ms', 'nedi_pixel_count',
    'nedi_bilinear_fallback_count', 'nedi_dimension_adjustment',
):
    print(f'{field}: {record[field]}')

print(f'Reconstructed image folder: {IMAGES_ROOT}')